# Guided model comparison — one notebook, one kernel

Run the numbered steps in order. **Edit only Step 1.** Keep the working `VGGT360 Comparison` kernel selected. Model environments are launched automatically in separate processes.

Completed model outputs are reused. Failed attempts can be archived automatically. GT and source images are never modified. Existing environments must be installed once using `docs/model_comparison.md`; this notebook checks them but does not install packages automatically.

To add another model: change MODEL, ENVIRONMENT and CHECKPOINT in Step 1, keep JOB_NAME and FRAME_IDS unchanged, then run the steps again. To select different frames, use a new JOB_NAME. Start with ACTION='view' to verify your existing result, then use ACTION='run' when ready.

## Step 1 — Settings (the only cell to edit)

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEQUENCE = ROOT / 'standard_GT' / 'city01_route01'
JOB_NAME = 'vggt360_smoke01'  # Reuses your successful VGGT-360 output.
FRAME_IDS = [0]

MODEL = 'vggt360'  # vggt360 | panovggt | omnivggt | mapanything
ACTION = 'view'   # view: display existing results | run: run only MODEL, then display all results
ENVIRONMENT = None  # None: notebook Python; otherwise conda name or full environment/Python path.
CHECKPOINT = None   # PanoVGGT: existing weight path. OmniVGGT: optional local weight path.
RETRY_INCOMPLETE = True  # Preserve a failed attempt in failed_attempts/ before retrying.
RAW_LIMITS = None  # Separate raw colorbars; optional shared numeric range, e.g. (0, 100).


## Step 2 — Validate settings and show the plan

A mismatch between saved frames and your settings stops here, without overwriting anything. Model selection does not change which saved results are displayed.

In [ ]:
import json
import pandas as pd
from IPython.display import display
from comparison.guided import validate_settings, prepare, status, preflight, run
assert ACTION in ('view', 'run'), "ACTION must be 'view' or 'run'."
assert JOB_NAME and Path(JOB_NAME).name == JOB_NAME and JOB_NAME not in ('.', '..'), 'Use a simple folder name for JOB_NAME.'
JOB = ROOT / 'outputs' / 'depth_comparison' / JOB_NAME
SEQUENCE, JOB = validate_settings(SEQUENCE, JOB, FRAME_IDS, MODEL)
print('Action:', ACTION, '| Selected model:', MODEL)
print('Job:', JOB)
print('Frames:', FRAME_IDS)
print('Environment:', ENVIRONMENT or sys.executable)
print('GT input: disabled. Existing successful predictions: preserved.')


## Step 3 — Prepare or reuse the shared images

In [ ]:
if ACTION == 'view' and not (JOB / 'manifest.json').exists():
    raise ValueError("No existing job to view. Set ACTION='run' to prepare a new job.")
manifest = prepare(SEQUENCE, JOB, FRAME_IDS)
display(pd.DataFrame(manifest['frames'])[['frame_id', 'source_frame_id', 'source_name']])
display(pd.DataFrame(status(JOB)))


## Step 4 — Check the model environment

This checks Python, CUDA and key dependencies in the selected environment. A completed model or view-only action does not need its environment loaded. A dependency failure stops execution here; share that error instead of changing other settings.

In [ ]:
if ACTION == 'run':
    preflight(JOB, MODEL, ENVIRONMENT, CHECKPOINT)
else:
    print('View-only mode: no model process will be launched.')


## Step 5 — Run the selected model or reuse its result

Only MODEL can run. Other model results remain available. An unsuccessful attempt is preserved, not deleted, when RETRY_INCOMPLETE is enabled. Full logs are saved under the job's logs/ folder. No automatic retry loop is used.

In [ ]:
if ACTION == 'run':
    run(JOB, MODEL, ENVIRONMENT, CHECKPOINT, RETRY_INCOMPLETE)
else:
    print('Inference skipped. Continue to the plots.')
display(pd.DataFrame(status(JOB)))


## Step 6 — Compare all completed depth maps

The first plot independently normalizes each map by its 2nd/98th percentiles: scale differences are hidden. The second shows raw values with explicit colorbars. These are qualitative outputs, not accuracy scores or verified metres.

OmniVGGT/MapAnything use our six-view perspective adaptation. VGGT-360 uses official per-panorama fusion of world-point norms. These protocols differ; see the metadata and comparison guide.

In [ ]:
import matplotlib.pyplot as plt
from comparison.view import load_results, depth_grid, trajectories
_, results = load_results(JOB)
if not results:
    print('No model has completed yet. Check Step 5 before interpreting the plots.')
else:
    display(pd.DataFrame([{'model': name, **meta} for name, (_, meta) in results.items()]))
    for normalized, filename in [(True, 'depth_normalized.png'), (False, 'depth_raw.png')]:
        fig = depth_grid(JOB, normalized=normalized, raw_limits=RAW_LIMITS)
        fig.savefig(JOB / filename, dpi=150, bbox_inches='tight')
        plt.show()


## Step 7 — Predicted camera paths

Each model uses its own first-camera frame and scale; no GT scoring or alignment. At least two selected frames are needed to draw movement. VGGT-360's virtual-view poses are not a temporal trajectory.

In [ ]:
if len(FRAME_IDS) < 2:
    print('One panorama selected: no temporal path to show.')
else:
    fig = trajectories(JOB)
    if fig is not None:
        fig.savefig(JOB / 'predicted_paths.png', dpi=150, bbox_inches='tight')
        plt.show()


## Step 8 — Save a shareable report

The ZIP includes metadata, plots and saved run logs. It contains neither weights nor original images. Run this step after a failed run too, if you want to share its log.

In [ ]:
import zipfile
from datetime import datetime, timezone
report_path = JOB / ('comparison_report_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ') + '.zip')
with zipfile.ZipFile(report_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    candidates = [JOB / 'manifest.json', *JOB.glob('*.png'), *JOB.glob('predictions/*/metadata.json'), *JOB.glob('logs/*.txt')]
    for path in candidates:
        if path.is_file(): archive.write(path, path.relative_to(JOB))
print('Share this report:', report_path)
